In [21]:
# conda activate chronocell

import os, sys
import numpy as np
import pandas as pd

sys.path.append("/mnt/lareaulab/reliscu/programs/FGP_2024")
sys.path.append("code")

import Chronocell
from reconstruct_RNA_history import *

In [22]:
# Get traj object from running Chronocell
import pickle
with open("eLNPs_var>1.2_traj_WS.pkl", "rb") as f:
    traj = pickle.load(f)

In [105]:
Y = traj.X
tau = traj.tau # State transition times (global)
t = traj.t
theta = traj.theta
topo = traj.topo

theta_ = theta.copy()
a0 = theta_[:2, 0] # Starting RNA abundance 
a = theta_[:2, 1:len(topo.flatten())] 
beta = theta_[:2, -2] # Splicing rate
alpha = a * beta[:2, None] # These values are divided by splicing rate; removing this factor now
gamma = theta_[:2, -1] # Degradation rate
state_grid = np.searchsorted(tau, t, side="right") - 1
states, index_for = enumerate_states(U_max, S_max)

In [104]:
np.searchsorted(tau, t, side="left") - 1

array([-1,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
        0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
        0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  1,  1,  1,  1,  1,  1,
        1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,
        1,  1,  1,  1,  1,  1,  1,  1,  2,  2,  2,  2,  2,  2,  2,  2,  2,
        2,  2,  2,  2,  2,  2,  2,  2,  2,  2,  2,  2,  2,  2,  2])

In [67]:
U_max = 20
S_max = 20
# np.sort(np.max(Y[:, :, 0], axis=0))

In [86]:
# Prep forward rate matrices

A_per_gene = [] 

for j in range(0, alpha.shape[0]):
    A_for_this_gene = []
    for i in range(0, alpha.shape[1]-1):
        rxns = define_reactions(alpha[j, i], beta[j], gamma[j])
        A1 = create_transition_matrix(rxns, U_max, S_max)
        A_for_this_gene.append(A1)
    A_per_gene.append(A_for_this_gene)

In [89]:
# Initialize X_fwd with stationary distribution (steady state at t=0)

pi_per_gene = []

for j in range(0, alpha.shape[0]):
    alpha0 = a0[j] * beta[j]
    rxns0 = define_reactions(alpha0, beta[j], gamma[j])
    A0 = create_transition_matrix(rxns0, U_max, S_max)
    pi = stationary_from_transition_matrix(A0)
    pi_per_gene.append(pi)

In [ ]:
X_fwd_list = []

for j in range(0, alpha.shape[0]):
    X_fwd = forward_distribution(A_per_gene[j], pi_per_gene[j], states, t, tau, state_grid)
    X_fwd_list.append(X_fwd)

662

In [ ]:
# j = 0
# X_fwd = forward_distribution(A_per_gene[j], pi_per_gene[j], states, t, tau, state_grid)

In [93]:
j = 0
A = A_per_gene[j]
pi = pi_per_gene[j]

In [ ]:
X_fwd = np.zeros(shape=(len(states), len(t)))
X_fwd[:, 0] = pi

for k in range(1, len(t)-1): 
    t_curr, t_next = t[k], t[k+1]
    state_curr, state_next = state_grid[k], state_grid[k+1]
    x_curr = X_fwd[:, k]
    
    if state_curr == state_next:
        dt = t_next - t_curr
        A_k = A[state_curr]
        x_next = x_curr @ sp.linalg.expm(A_k * dt) 
        
    else:   
        # State switch happens in current interval
        t_s = tau[state_next]

        # Split backward march into 2 steps
        dt1 = t_s - t_curr # left interval: [t_k, state_switch_time)
        A_k1 = A[state_curr]
        x_mid = x_curr @ sp.linalg.expm(A_k1 * dt1)
    
        dt2 = t_next - t_s # right interval: [state_switch_time, t_{k+1})
        A_k2 = A[state_next]
        x_next = x_mid @ sp.linalg.expm(A_k2 * dt2) 
    
    X_fwd[:, k+1] = x_next
    

IndexError: list index out of range

In [98]:
state_grid

array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2,
       2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 3])

In [97]:
len(A)

2